# 02 - Limpieza REE

Carga y limpieza de la generación fotovoltaica nacional diaria y por CCAA, y cálculo de los pesos de agregación espacial usados en `03_integracion_datos`.

## 1. Carga de datos

In [157]:
import pandas as pd
import numpy as np

ree = pd.read_csv(
    "../data/raw/ree/ree_fotovoltaica_2021_2025.csv"
)

ree_ccaa = pd.read_csv(
    "../data/raw/ree/ree_fotovoltaica_ccaa_2021_2025.csv"
)

print(ree.shape)
print(ree.head())
print(ree.info())
print(ree.columns.tolist())

(1825, 3)
       value  percentage                   datetime
0  34247.163    0.021989  2021-01-01 23:00:00+00:00
1  37134.485    0.024855  2021-01-02 23:00:00+00:00
2  26307.981    0.017347  2021-01-03 23:00:00+00:00
3  29746.352    0.020739  2021-01-04 23:00:00+00:00
4  25273.930    0.018880  2021-01-05 23:00:00+00:00
<class 'pandas.DataFrame'>
RangeIndex: 1825 entries, 0 to 1824
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   value       1825 non-null   float64
 1   percentage  1825 non-null   float64
 2   datetime    1825 non-null   str    
dtypes: float64(2), str(1)
memory usage: 87.5 KB
None
['value', 'percentage', 'datetime']


In [158]:
print(ree_ccaa.shape)
print(ree_ccaa.head())
print(ree_ccaa.info())
print(ree_ccaa.columns.tolist())

(1097, 5)
        value  percentage                       datetime       ccaa  id_ccaa
0  213710.908    0.052033  2021-01-01T00:00:00.000+01:00  Andalucia        4
1  231989.192    0.065533  2021-02-01T00:00:00.000+01:00  Andalucia        4
2  408278.780    0.096378  2021-03-01T00:00:00.000+01:00  Andalucia        4
3  402297.899    0.088198  2021-04-01T00:00:00.000+02:00  Andalucia        4
4  556125.915    0.111763  2021-05-01T00:00:00.000+02:00  Andalucia        4
<class 'pandas.DataFrame'>
RangeIndex: 1097 entries, 0 to 1096
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   value       1097 non-null   float64
 1   percentage  1097 non-null   float64
 2   datetime    1097 non-null   str    
 3   ccaa        1097 non-null   str    
 4   id_ccaa     1097 non-null   int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 85.4 KB
None
['value', 'percentage', 'datetime', 'ccaa', 'id_ccaa']


## 2. Exploración de nulos y duplicados

In [159]:
print(
    ree.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

value         0.0
percentage    0.0
datetime      0.0
dtype: float64


In [160]:
print(
    ree_ccaa.isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

value         0.0
percentage    0.0
datetime      0.0
ccaa          0.0
id_ccaa       0.0
dtype: float64


In [161]:
print("Duplicados completos:", ree.duplicated().sum())
print("Duplicados por fecha:", ree.duplicated(subset=["datetime"]).sum())

print("Duplicados completos:", ree_ccaa.duplicated().sum())

Duplicados completos: 0
Duplicados por fecha: 0
Duplicados completos: 0


## 3. Tipado de columnas

In [162]:
ree["datetime"] = pd.to_datetime(ree["datetime"], utc=True, errors="coerce")
ree["datetime"] = ree["datetime"].dt.tz_convert("Europe/Madrid").dt.normalize().dt.tz_localize(None)
ree["value"] = pd.to_numeric(ree["value"], errors="coerce")
ree["percentage"] = pd.to_numeric(ree["percentage"], errors="coerce")

ree_ccaa["datetime"] = pd.to_datetime(ree_ccaa["datetime"], utc=True, errors="coerce")
ree_ccaa["datetime"] = ree_ccaa["datetime"].dt.tz_convert("Europe/Madrid").dt.normalize().dt.tz_localize(None)
ree_ccaa["value"] = pd.to_numeric(ree_ccaa["value"], errors="coerce")
ree_ccaa["percentage"] = pd.to_numeric(ree_ccaa["percentage"], errors="coerce")

# Verificacion: ninguna fecha deberia haberse perdido en el parseo
print("NaT en ree:", ree["datetime"].isna().sum())
print("NaT en ree_ccaa:", ree_ccaa["datetime"].isna().sum())
print(ree["datetime"].dtype, ree_ccaa["datetime"].dtype)

NaT en ree: 0
NaT en ree_ccaa: 0
datetime64[us] datetime64[us]


## 4. Detección de valores anómalos

In [163]:
print("Generacion negativa:", (ree["value"] < 0).sum())
print("Porcentaje negativo:", (ree["percentage"] < 0).sum())
print("Porcentaje > 100:", (ree["percentage"] > 100).sum())

Generacion negativa: 0
Porcentaje negativo: 0
Porcentaje > 100: 0


In [164]:
print("Generacion negativa:", (ree_ccaa["value"] < 0).sum())
print("Porcentaje negativo:", (ree_ccaa["percentage"] < 0).sum())
print("Porcentaje > 100:", (ree_ccaa["percentage"] > 100).sum())

Generacion negativa: 0
Porcentaje negativo: 0
Porcentaje > 100: 0


## 5. Validación de fechas

In [165]:
print("Fecha minima:", ree["datetime"].min())
print("Fecha maxima:", ree["datetime"].max())
print(ree["datetime"].dt.year.value_counts().sort_index())

Fecha minima: 2021-01-02 00:00:00
Fecha maxima: 2025-12-31 00:00:00
datetime
2021    364
2022    365
2023    365
2024    366
2025    365
Name: count, dtype: int64


In [166]:
print("Fecha minima:", ree_ccaa["datetime"].min())
print("Fecha maxima:", ree_ccaa["datetime"].max())
print(ree_ccaa["datetime"].dt.year.value_counts().sort_index())

Fecha minima: 2021-01-01 00:00:00
Fecha maxima: 2025-12-01 00:00:00
datetime
2021    215
2022    216
2023    220
2024    224
2025    222
Name: count, dtype: int64


In [167]:
print(ree_ccaa["ccaa"].unique())

<ArrowStringArray>
[           'Andalucia',               'Aragon',             'Asturias',
            'Cantabria',      'Castilla_y_Leon',   'Castilla_La_Mancha',
             'Cataluna', 'Comunitat_Valenciana',          'Extremadura',
              'Galicia',        'Illes_Balears',             'Canarias',
             'La_Rioja',     'Comunidad_Madrid',        'Region_Murcia',
              'Navarra',           'Pais_Vasco',                'Ceuta',
              'Melilla']
Length: 19, dtype: str


## 6. Renombrado y orden final

In [168]:
ree = ree.rename(columns={
    "datetime": "fecha",
    "value": "generacion_fotovoltaica",
    "percentage": "porcentaje_fotovoltaica"
})

ree_ccaa = ree_ccaa.rename(columns={
    "datetime": "fecha",
    "value": "generacion_fotovoltaica",
    "percentage": "porcentaje_fotovoltaica"
})

ree = ree.sort_values("fecha").reset_index(drop=True)
ree_ccaa = ree_ccaa.sort_values(["ccaa", "fecha"]).reset_index(drop=True)

In [ ]:

ree_ccaa["ccaa"] = ree_ccaa["ccaa"].str.replace("_", " ", regex=False)

print(sorted(ree_ccaa["ccaa"].unique()))

['Andalucia', 'Aragon', 'Asturias', 'Canarias', 'Cantabria', 'Castilla La Mancha', 'Castilla y Leon', 'Cataluna', 'Ceuta', 'Comunidad Madrid', 'Comunitat Valenciana', 'Extremadura', 'Galicia', 'Illes Balears', 'La Rioja', 'Melilla', 'Navarra', 'Pais Vasco', 'Region Murcia']


## 7. Normalización de nombres de CCAA

Los nombres que devuelve la API de REE (con tildes, artículos, etc.) deben coincidir exactamente con los valores usados como destino en el diccionario `provincia_a_ccaa` de `01_limpieza_aemet`, o el merge de pesos en `03_integracion_datos` dejará esas filas sin peso (`NaN`) sin avisar. Revisa la salida de `ree_ccaa["ccaa"].unique()` de la celda anterior y ajusta este diccionario si aparece alguna variante no contemplada.

In [170]:
normalizacion_ccaa = {
    "Andalucía": "Andalucia",
    "Aragón": "Aragon",
    "Principado de Asturias": "Asturias",
    "Asturias, Principado de": "Asturias",
    "Illes Balears": "Baleares",
    "Islas Baleares": "Baleares",
    "Canarias": "Canarias",
    "Islas Canarias": "Canarias",
    "Cantabria": "Cantabria",
    "Castilla y León": "Castilla y Leon",
    "Castilla-La Mancha": "Castilla la Mancha",
    "Castilla La Mancha": "Castilla la Mancha",
    "Cataluña": "Cataluna",
    "Comunidad Valenciana": "Comunidad Valenciana",
    "Comunitat Valenciana": "Comunidad Valenciana",
    "Extremadura": "Extremadura",
    "Galicia": "Galicia",
    "Comunidad de Madrid": "Madrid",
    "Comunidad Madrid": "Madrid",
    "Madrid, Comunidad de": "Madrid",
    "Región de Murcia": "Murcia",
    "Region Murcia": "Murcia",
    "Murcia, Región de": "Murcia",
    "Comunidad Foral de Navarra": "Navarra",
    "Comunidad de Navarra": "Navarra",
    "País Vasco": "Pais Vasco",
    "La Rioja": "La Rioja",
    "Rioja, La": "La Rioja",
    "Ceuta": "Ceuta",
    "Melilla": "Melilla"
}

ree_ccaa["ccaa"] = ree_ccaa["ccaa"].replace(normalizacion_ccaa)

valores_esperados = set(normalizacion_ccaa.values())
sin_normalizar = set(ree_ccaa["ccaa"].unique()) - valores_esperados
print("Valores de ccaa no reconocidos tras normalizar:", sin_normalizar)
print("Total CCAA tras normalizar:", ree_ccaa["ccaa"].nunique())

Valores de ccaa no reconocidos tras normalizar: set()
Total CCAA tras normalizar: 19


## 8. Cobertura temporal por CCAA y corrección de huecos 

In [ ]:


ree_ccaa["año"] = ree_ccaa["fecha"].dt.year

# Cobertura de meses por año, para TODAS las CCAA a la vez
cobertura_completa = (
    ree_ccaa
    .groupby(["ccaa", "año"])
    .size()
    .reset_index(name="meses_disponibles")
)

print(
    cobertura_completa
    .pivot(index="ccaa", columns="año", values="meses_disponibles")
    .fillna(0)
    .astype(int)
)

# Ceuta: hay meses sueltos sin fotovoltaica reportada (confirmado contra la
# API, no es un hueco de descarga). Se identifican los meses que faltan
# comparando contra el calendario completo del periodo 2021-2025 y se
# rellenan con generacion = 0.

ceuta = ree_ccaa[ree_ccaa["ccaa"] == "Ceuta"].copy()

rango_completo_ceuta = pd.date_range("2021-01-01", ree_ccaa["fecha"].max(), freq="MS")
fechas_presentes_ceuta = set(ceuta["fecha"])

meses_faltantes_ceuta = [
    f for f in rango_completo_ceuta if f not in fechas_presentes_ceuta
]

print("Meses de Ceuta a rellenar con 0:", len(meses_faltantes_ceuta))
print(meses_faltantes_ceuta)

relleno_ceuta = pd.DataFrame({
    "fecha": meses_faltantes_ceuta,
    "generacion_fotovoltaica": 0.0,
    "porcentaje_fotovoltaica": 0.0,
    "ccaa": "Ceuta",
})
relleno_ceuta["año"] = relleno_ceuta["fecha"].dt.year

ree_ccaa = pd.concat([ree_ccaa, relleno_ceuta], ignore_index=True)
ree_ccaa = ree_ccaa.sort_values(["ccaa", "fecha"]).reset_index(drop=True)


print(
    ree_ccaa[ree_ccaa["ccaa"] == "Ceuta"]
    .groupby("año")
    .size()
)

año                   2021  2022  2023  2024  2025
ccaa                                              
Andalucia               12    12    12    12    12
Aragon                  12    12    12    12    12
Asturias                12    12    12    12    12
Baleares                12    12    12    12    12
Canarias                12    12    12    12    12
Cantabria               12    12    12    12    12
Castilla la Mancha      12    12    12    12    12
Castilla y Leon         12    12    12    12    12
Cataluna                12    12    12    12    12
Ceuta                    0     0     4     8     6
Comunidad Valenciana    12    12    12    12    12
Extremadura             12    12    12    12    12
Galicia                 12    12    12    12    12
La Rioja                12    12    12    12    12
Madrid                  12    12    12    12    12
Melilla                 11    12    12    12    12
Murcia                  12    12    12    12    12
Navarra                 12    1

## 9. Cálculo de pesos por CCAA y año

Se usa la generación fotovoltaica **anual** agregada por CCAA como proxy de la potencia instalada: a escala anual el efecto de "más o menos sol ese año en una región concreta" se diluye, por lo que la proporción de generación entre CCAA se aproxima razonablemente al reparto real de potencia instalada. Es una simplificación frente a usar potencia instalada real (no disponible en los datos actuales) que conviene documentar como tal en la memoria del TFM.

In [172]:
generacion_anual_ccaa = (
    ree_ccaa.groupby(["año", "ccaa"])["generacion_fotovoltaica"]
    .sum()
    .reset_index(name="generacion_anual")
)

total_anual = (
    generacion_anual_ccaa.groupby("año")["generacion_anual"]
    .sum()
    .reset_index(name="generacion_total_año")
)

pesos = generacion_anual_ccaa.merge(total_anual, on="año")
pesos["peso"] = pesos["generacion_anual"] / pesos["generacion_total_año"]
pesos = pesos[["año", "ccaa", "peso"]].sort_values(["año", "peso"], ascending=[True, False])

pesos.head(20)

,año,ccaa,peso
11,2021,Extremadura,0.235391
0,2021,Andalucia,0.234290
6,2021,Castilla la Mancha,0.178602
16,2021,Murcia,0.095776
1,2021,Aragon,0.091871
7,2021,Castilla y Leon,0.070839
10,2021,Comunidad Valenciana,0.024729
8,2021,Cataluna,0.018078
17,2021,Navarra,0.013830
4,2021,Canarias,0.012519


In [ ]:

check_suma = pesos.groupby("año")["peso"].sum()
print(check_suma)

assert (abs(check_suma - 1.0) < 1e-6).all(), "Los pesos de algun año no suman 1"
print("OK: los pesos de todos los años suman 1")

año
2021    1.0
2022    1.0
2023    1.0
2024    1.0
2025    1.0
Name: peso, dtype: float64
OK: los pesos de todos los años suman 1


In [ ]:

top_ccaa = (
    pesos.groupby("ccaa")["peso"].mean()
    .sort_values(ascending=False)
    .head(5)
    .index
)

pesos[pesos["ccaa"].isin(top_ccaa)].pivot(index="año", columns="ccaa", values="peso").round(3)

ccaa,Andalucia,Aragon,Castilla la Mancha,Castilla y Leon,Extremadura
año,,,,,
2021,0.234,0.092,0.179,0.071,0.235
2022,0.210,0.102,0.214,0.071,0.249
2023,0.227,0.103,0.228,0.069,0.245
2024,0.232,0.093,0.243,0.077,0.233
2025,0.264,0.086,0.223,0.096,0.214


## 10. Guardado de salidas

In [175]:
ree.to_csv(
    "../data/processed/ree_fotovoltaica_diario_limpio_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

ree_ccaa.to_csv(
    "../data/processed/ree_fotovoltaica_ccaa_mensual_limpio_2021_2025.csv",
    index=False,
    encoding="utf-8-sig"
)

pesos.to_csv(
    "../data/processed/pesos_ccaa_por_año.csv",
    index=False,
    encoding="utf-8-sig"
)


pesos.to_parquet(
    "../data/processed/pesos_ccaa_por_año.parquet",
    index=False
)

print("Guardado ree:", ree.shape)
print("Guardado ree_ccaa:", ree_ccaa.shape)
print("Guardado pesos:", pesos.shape)

Guardado ree: (1825, 3)
Guardado ree_ccaa: (1139, 6)
Guardado pesos: (95, 3)
